In [3]:
# --- 0. 필요한 라이브러리 및 클래스 정의 ---
import pandas as pd
import re
import os
import joblib
from typing import List
from kiwipiepy import Kiwi

class KiwiTokenizer:

    def __init__(self):
        self.kiwi = None
        self.stop_words = ["하", "있", "되"]

    def __call__(self, text: str) -> List[str]:
        if self.kiwi is None:
            self.kiwi = Kiwi()
            self.kiwi.add_user_word("대포통장", "NNP")
            self.kiwi.add_user_word("계좌번호", "NNP")

        cleaned_text = re.sub(r"[^\w\s<>]", " ", text)
        cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

        tokens = self.kiwi.tokenize(cleaned_text)
        result_tokens = [
            token.form
            for token in tokens
            if token.tag in ["NNG", "NNP", "VV", "VA", "XR"]
            and token.form not in self.stop_words
        ]
        return result_tokens

    def __getstate__(self):
        # 저장 시 호출: kiwi 객체는 저장하지 않음
        state = self.__dict__.copy()
        if "kiwi" in state:
            del state["kiwi"]
        return state

    def __setstate__(self, state):
        # 로드 시 호출: kiwi 객체를 None으로 초기화
        self.__dict__.update(state)
        self.kiwi = None


print("초기 설정 및 필수 클래스 정의 완료.")

초기 설정 및 필수 클래스 정의 완료.


# 로지스틱 회귀 K-Fold

In [4]:
# --- 0. 필요한 라이브러리 및 클래스 정의 ---
import pandas as pd
import re
import os
import joblib
from typing import List
from kiwipiepy import Kiwi
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# 윈도우 환경에서 한글 폰트가 깨지지 않도록 설정
plt.rcParams["font.family"] = "Malgun Gothic"
# 마이너스 기호가 깨지는 문제 해결
plt.rcParams["axes.unicode_minus"] = False


class KiwiTokenizer:

    def __init__(self):
        self.kiwi = None
        self.stop_words = ["하", "있", "되"]

    def __call__(self, text: str) -> List[str]:
        # 각 프로세스에서 최초 호출 시 Kiwi 객체를 한 번만 생성
        if self.kiwi is None:
            self.kiwi = Kiwi()
            self.kiwi.add_user_word("대포통장", "NNP")
            self.kiwi.add_user_word("계좌번호", "NNP")

        # 텍스트 정제
        cleaned_text = re.sub(r"[^\w\s<>]", " ", text)
        cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

        # 형태소 분석 및 필터링
        tokens = self.kiwi.tokenize(cleaned_text)
        result_tokens = [
            token.form
            for token in tokens
            if token.tag in ["NNG", "NNP", "VV", "VA", "XR"]
            and token.form not in self.stop_words
        ]
        return result_tokens

    def __getstate__(self):
        # 저장 시 호출: kiwi 객체는 저장하지 않음
        state = self.__dict__.copy()
        if "kiwi" in state:
            del state["kiwi"]
        return state

    def __setstate__(self, state):
        # 로드 시 호출: kiwi 객체를 None으로 초기화
        self.__dict__.update(state)
        self.kiwi = None


print("초기 설정 및 최종 클래스 정의 완료.")

초기 설정 및 최종 클래스 정의 완료.


In [5]:
# --- 1. 파일 경로 및 모델/Vectorizer 로딩 ---
print("저장된 Vectorizer와 Logistic Regression 모델을 불러옵니다...")

# Vectorizer 파일 경로
vectorizer_path = os.path.join("tfidf_vectorizer_6000_JJ.pkl")
# 로지스틱 회귀 모델 파일 경로
lr_model_path = os.path.join("models", "logistic_regression_model_6000_JJ.pkl")

try:
    loaded_vectorizer = joblib.load(vectorizer_path)
    loaded_lr_model = joblib.load(lr_model_path)
    print(f"Vectorizer 로드 완료: {vectorizer_path}")
    print(f"Logistic Regression 모델 로드 완료: {lr_model_path}")

except Exception as e:
    print(f"파일 로딩 중 에러 발생: {e}")

저장된 Vectorizer와 Logistic Regression 모델을 불러옵니다...
파일 로딩 중 에러 발생: Can't get attribute 'my_kiwi_tokenizer' on <module '__main__'>
